# Adversarial CIFAR-10 Classifier
Trains three models (Standard CNN, Whitebox-PGD, Blackbox-PGD), compares robustness, and visualises PGD attack examples.

In [13]:
# ── Configuration ─────────────────────────────────────────────────────────────
EPOCHS          = 30    # standard training epochs
ADV_EPOCHS      = 20    # adversarial training epochs (slower per-epoch)
BATCH_SIZE      = 128
LR              = 0.1
PGD_EPS         = 8 / 255
PGD_ALPHA       = 2 / 255
PGD_STEPS_TRAIN = 7
PGD_STEPS_EVAL  = 20
N_ATTACK_IMGS   = 5     # original-vs-attacked pairs to save
DATA_ROOT       = './data'
ARCHIVE_PATH    = './data/cifar-10-python.tar.gz'  # local archive; set None to force download
ARTIFACTS       = './artifacts'
SEED            = 42

In [14]:
# ── Imports & device ──────────────────────────────────────────────────────────
import csv
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')

for d in ['checkpoints', 'plots', 'attack_examples']:
    Path(f'{ARTIFACTS}/{d}').mkdir(parents=True, exist_ok=True)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

Device: cuda


## Model

In [15]:
class SimpleCNN(nn.Module):
    """Three conv blocks + global-avg-pool + dense head. NCHW input."""

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   128, 3, padding=1), nn.GroupNorm(16, 128),  nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.GroupNorm(16, 128),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.GroupNorm(32, 256),  nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.GroupNorm(32, 256),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.GroupNorm(64, 512),  nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.GroupNorm(64, 512),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.head = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).mean(dim=[2, 3]))

## PGD Attack

In [16]:
def pgd_linf_attack(
    model: nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    epsilon: float,
    alpha: float,
    num_steps: int,
    random_start: bool = True,
) -> torch.Tensor:
    """Untargeted PGD L-inf. Loss: -mean(sum(one_hot * softmax(logits)))."""
    was_training = model.training
    model.eval()

    x = images.clone().detach()
    if random_start:
        x = torch.clamp(x + torch.empty_like(x).uniform_(-epsilon, epsilon), 0.0, 1.0)
    original = images.clone().detach()

    for _ in range(num_steps):
        x = x.detach().requires_grad_(True)
        loss = -torch.mean(
            torch.sum(F.one_hot(labels, 10).float() * F.softmax(model(x), dim=1), dim=1)
        )
        loss.backward()
        with torch.no_grad():
            x = torch.clamp(
                original + torch.clamp(x + alpha * x.grad.sign() - original, -epsilon, epsilon),
                0.0, 1.0,
            )

    if was_training:
        model.train()
    return x.detach()

## Data

In [17]:
import tarfile

def build_loaders(data_root, batch_size, num_workers=0, download=True,
                  archive_path=None, val_fraction=0.1, seed=42):
    root      = Path(data_root)
    cifar_dir = root / 'cifar-10-batches-py'
    root.mkdir(parents=True, exist_ok=True)

    if not cifar_dir.exists():
        # 1. Try extracting from a local archive
        ap = Path(archive_path) if archive_path else root / 'cifar-10-python.tar.gz'
        if ap.exists():
            print(f'Extracting CIFAR-10 from {ap} ...')
            with tarfile.open(ap, 'r:*') as tar:
                tar.extractall(path=root)
            print('Extraction complete.')
        elif download:
            # 2. Download from the web (network must be available)
            print('Local archive not found; downloading CIFAR-10 from the web...')
        else:
            raise FileNotFoundError(
                f'CIFAR-10 not found at {root}.\n'
                'Options: (a) set ARCHIVE_PATH to your local tar.gz, '
                'or (b) set download=True to fetch from the web.'
            )

    # Only ask torchvision to download if the data is still missing
    need_download = download and not cifar_dir.exists()

    aug   = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.ToTensor()])
    plain = transforms.ToTensor()
    aug_ds   = datasets.CIFAR10(str(root), train=True,  transform=aug,   download=need_download)
    plain_ds = datasets.CIFAR10(str(root), train=True,  transform=plain, download=False)
    test_ds  = datasets.CIFAR10(str(root), train=False, transform=plain, download=False)

    idx   = np.random.default_rng(seed).permutation(len(aug_ds))
    n_val = int(len(aug_ds) * val_fraction)
    pin   = torch.cuda.is_available()
    kw    = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=pin)
    return (
        DataLoader(Subset(aug_ds,   idx[n_val:]), shuffle=True,  **kw),
        DataLoader(Subset(plain_ds, idx[:n_val]), shuffle=False, **kw),
        DataLoader(test_ds,                       shuffle=False, **kw),
    )

train_loader, val_loader, test_loader = build_loaders(
    DATA_ROOT, BATCH_SIZE, archive_path=ARCHIVE_PATH
)
print(f'Train: {len(train_loader.dataset):,}  '
      f'Val: {len(val_loader.dataset):,}  '
      f'Test: {len(test_loader.dataset):,}')

Local archive not found; downloading CIFAR-10 from the web...


HTTPError: HTTP Error 503: Service Unavailable

## Training Utilities

In [ ]:
def mse_loss(logits, labels):
    return F.mse_loss(F.softmax(logits, dim=1), F.one_hot(labels, 10).float())


def train_epoch(model, loader, optimizer, mode, surrogate=None):
    model.train()
    total_loss = total_correct = total = 0
    for imgs, lbs in tqdm(loader, desc='train', leave=False):
        imgs, lbs = imgs.to(device), lbs.to(device)
        if mode == 'whitebox_pgd':
            imgs = pgd_linf_attack(model, imgs, lbs, PGD_EPS, PGD_ALPHA, PGD_STEPS_TRAIN)
            model.train()
        elif mode == 'blackbox_pgd':
            imgs = pgd_linf_attack(surrogate, imgs, lbs, PGD_EPS, PGD_ALPHA, PGD_STEPS_TRAIN)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = mse_loss(logits, lbs)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += logits.argmax(1).eq(lbs).sum().item()
        total         += imgs.size(0)
    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total = 0
    all_probs, all_labels = [], []
    for imgs, lbs in loader:
        imgs, lbs = imgs.to(device), lbs.to(device)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1)
        total_loss += mse_loss(logits, lbs).item() * imgs.size(0)
        total      += imgs.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(lbs.cpu())
    probs  = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    acc = (probs.argmax(1) == labels).float().mean().item()
    try:
        auc = float(roc_auc_score(F.one_hot(labels, 10).numpy(), probs.numpy(),
                                   average='macro', multi_class='ovr'))
    except Exception:
        auc = float('nan')
    return total_loss / max(total, 1), acc, auc


def train_model(name, model, optimizer, epochs, mode='standard', surrogate=None):
    ckpt     = Path(f'{ARTIFACTS}/checkpoints/{name}.pt')
    csv_path = Path(f'{ARTIFACTS}/{name}_history.csv')
    history, best_acc = [], -1.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, mode, surrogate)
        vl_loss, vl_acc, vl_auc = evaluate(model, val_loader)
        te_loss, te_acc, te_auc = evaluate(model, test_loader)
        row = dict(epoch=epoch,
                   train_loss=tr_loss, train_acc=tr_acc,
                   val_loss=vl_loss,   val_acc=vl_acc,   val_auc=vl_auc,
                   test_loss=te_loss,  test_acc=te_acc,  test_auc=te_auc)
        history.append(row)
        print(f'[{name}] {epoch:3d}/{epochs}  '
              f'train={tr_acc*100:.1f}%  val={vl_acc*100:.1f}%  test={te_acc*100:.1f}%')
        if te_acc > best_acc:
            best_acc = te_acc
            torch.save(model.state_dict(), ckpt)

    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        w.writeheader()
        w.writerows(history)

    print(f'[{name}] Best test acc: {best_acc*100:.2f}%  →  {ckpt}')
    return history

## Train Standard Model

In [ ]:
standard_model = SimpleCNN().to(device)
opt_std = torch.optim.SGD(standard_model.parameters(), lr=LR, momentum=0.9, nesterov=True)
history_standard = train_model('standard', standard_model, opt_std, EPOCHS, mode='standard')

standard_model.load_state_dict(
    torch.load(f'{ARTIFACTS}/checkpoints/standard.pt', map_location=device)
)
standard_model.eval()
print('Standard model: best checkpoint loaded.')

## Train Whitebox-PGD Model

In [ ]:
whitebox_model = SimpleCNN().to(device)
opt_wb = torch.optim.SGD(whitebox_model.parameters(), lr=LR, momentum=0.9, nesterov=True)
history_whitebox = train_model('whitebox_pgd', whitebox_model, opt_wb,
                               ADV_EPOCHS, mode='whitebox_pgd')

whitebox_model.load_state_dict(
    torch.load(f'{ARTIFACTS}/checkpoints/whitebox_pgd.pt', map_location=device)
)
whitebox_model.eval()
print('Whitebox-PGD model: best checkpoint loaded.')

## Train Blackbox-PGD Model

In [ ]:
# Uses standard_model as the surrogate attacker
blackbox_model = SimpleCNN().to(device)
opt_bb = torch.optim.SGD(blackbox_model.parameters(), lr=LR, momentum=0.9, nesterov=True)
history_blackbox = train_model('blackbox_pgd', blackbox_model, opt_bb,
                               ADV_EPOCHS, mode='blackbox_pgd', surrogate=standard_model)

blackbox_model.load_state_dict(
    torch.load(f'{ARTIFACTS}/checkpoints/blackbox_pgd.pt', map_location=device)
)
blackbox_model.eval()
print('Blackbox-PGD model: best checkpoint loaded.')

## Training Curves

In [ ]:
def plot_history(history, title, save_path):
    ep = [r['epoch'] for r in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    for ax, key, ylabel in [(ax1, 'acc', 'Accuracy (%)'), (ax2, 'loss', 'Loss')]:
        scale = 100 if key == 'acc' else 1
        ax.plot(ep, [r[f'train_{key}'] * scale for r in history], 'o-', label='Train')
        ax.plot(ep, [r[f'val_{key}']   * scale for r in history], 's-', label='Val')
        ax.plot(ep, [r[f'test_{key}']  * scale for r in history], '^-', label='Test')
        ax.set(xlabel='Epoch', ylabel=ylabel,
               title=f'{title} – {ylabel.split(" ")[0]}')
        if key == 'acc':
            ax.set_ylim(0, 105)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: {save_path}')


plot_history(history_standard, 'Standard CNN',
             f'{ARTIFACTS}/plots/standard_training.png')
plot_history(history_whitebox, 'Whitebox-PGD CNN',
             f'{ARTIFACTS}/plots/whitebox_pgd_training.png')
plot_history(history_blackbox, 'Blackbox-PGD CNN',
             f'{ARTIFACTS}/plots/blackbox_pgd_training.png')

## Evaluate All Models

In [ ]:
def eval_under_attack(victim, attacker, loader):
    victim.eval()
    total_correct = total = 0
    for imgs, lbs in tqdm(loader, desc='adv eval', leave=False):
        imgs, lbs = imgs.to(device), lbs.to(device)
        adv = pgd_linf_attack(attacker, imgs, lbs, PGD_EPS, PGD_ALPHA, PGD_STEPS_EVAL)
        total_correct += victim(adv).argmax(1).eq(lbs).sum().item()
        total         += imgs.size(0)
    return total_correct / max(total, 1)


models_dict = {
    'Standard CNN':  standard_model,
    'Whitebox PGD':  whitebox_model,
    'Blackbox PGD':  blackbox_model,
}

results = {}
for name, model in models_dict.items():
    print(f'Evaluating {name}...')
    _, clean_acc, clean_auc = evaluate(model, test_loader)
    wb_acc = eval_under_attack(model, model,          test_loader)
    tf_acc = eval_under_attack(model, standard_model, test_loader)
    results[name] = dict(clean_acc=clean_acc, whitebox_pgd_acc=wb_acc,
                         transfer_pgd_acc=tf_acc, auc=clean_auc)
    print(f'  clean={clean_acc*100:.2f}%  '
          f'wb_pgd={wb_acc*100:.2f}%  '
          f'transfer={tf_acc*100:.2f}%  '
          f'AUC={clean_auc*100:.2f}%')

# Save CSV
with open(f'{ARTIFACTS}/comparison.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['model','clean_acc','whitebox_pgd_acc',
                                       'transfer_pgd_acc','auc'])
    w.writeheader()
    for name, r in results.items():
        w.writerow({'model': name, **r})
print(f'Saved: {ARTIFACTS}/comparison.csv')

## Comparison Bar Chart

In [ ]:
model_names = list(results.keys())
clean_accs  = [results[n]['clean_acc']        * 100 for n in model_names]
wb_accs     = [results[n]['whitebox_pgd_acc'] * 100 for n in model_names]
tf_accs     = [results[n]['transfer_pgd_acc'] * 100 for n in model_names]

x = np.arange(len(model_names))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w, clean_accs, w, label='Clean accuracy',          color='steelblue')
b2 = ax.bar(x,     wb_accs,    w, label=f'Whitebox PGD (ε=8/255)', color='tomato')
b3 = ax.bar(x + w, tf_accs,    w, label='Transfer PGD (from std)', color='darkorange')

for bars in (b1, b2, b3):
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)

ax.set(xticks=x, xticklabels=model_names, ylabel='Accuracy (%)',
       title=f'CIFAR-10 Robustness Comparison  (ε = {PGD_EPS:.4f}, '
             f'{PGD_STEPS_EVAL}-step PGD eval)')
ax.set_ylim(0, 115)
ax.legend(loc='upper right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()

save_path = f'{ARTIFACTS}/plots/comparison.png'
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {save_path}')

## Original vs PGD-Attacked Images

In [ ]:
# Pick one test image per class for the first N_ATTACK_IMGS classes
test_ds_plain = datasets.CIFAR10(DATA_ROOT, train=False,
                                  transform=transforms.ToTensor(), download=False)
class_imgs, class_lbs = {}, {}
for img, lb in test_ds_plain:
    if lb not in class_imgs:
        class_imgs[lb] = img
        class_lbs[lb]  = lb
    if len(class_imgs) >= N_ATTACK_IMGS:
        break

imgs_orig = torch.stack([class_imgs[c] for c in range(N_ATTACK_IMGS)]).to(device)
lbs       = torch.tensor([class_lbs[c]  for c in range(N_ATTACK_IMGS)], device=device)

# Generate PGD-20 attacks with the standard model
imgs_adv = pgd_linf_attack(standard_model, imgs_orig, lbs,
                            PGD_EPS, PGD_ALPHA, PGD_STEPS_EVAL)

# Show original model predictions vs attacked
with torch.no_grad():
    pred_orig = standard_model(imgs_orig).argmax(1).cpu()
    pred_adv  = standard_model(imgs_adv).argmax(1).cpu()

def to_np(t):
    return t.cpu().permute(1, 2, 0).clamp(0, 1).numpy()

fig, axes = plt.subplots(3, N_ATTACK_IMGS, figsize=(3 * N_ATTACK_IMGS, 9))

row_labels = ['Original', 'PGD attacked', 'Perturbation ×10']

for i in range(N_ATTACK_IMGS):
    orig    = to_np(imgs_orig[i])
    adv     = to_np(imgs_adv[i])
    diff    = np.clip((adv - orig) * 10 + 0.5, 0, 1)  # amplify & centre at grey

    true_cls  = CIFAR10_CLASSES[lbs[i].item()]
    orig_pred = CIFAR10_CLASSES[pred_orig[i].item()]
    adv_pred  = CIFAR10_CLASSES[pred_adv[i].item()]

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'{true_cls}\n→ {orig_pred}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(adv)
    color = 'red' if adv_pred != true_cls else 'green'
    axes[1, i].set_title(f'→ {adv_pred}', fontsize=9, color=color)
    axes[1, i].axis('off')

    axes[2, i].imshow(diff)
    axes[2, i].set_title('δ ×10', fontsize=9)
    axes[2, i].axis('off')

# Row labels via figure text
fig.subplots_adjust(left=0.10)
for row, label in enumerate(row_labels):
    # Normalised figure y-coordinate for each row centre (3 rows, equal height)
    y = 1 - (row + 0.55) / 3
    fig.text(0.01, y, label, fontsize=11, fontweight='bold',
             va='center', ha='left')

fig.suptitle(
    f'Original vs PGD-Attacked CIFAR-10  '
    f'(ε = {int(PGD_EPS*255)}/255, {PGD_STEPS_EVAL} steps)\n'
    f'Prediction colour: green = correct, red = fooled',
    fontsize=12, y=1.01,
)
plt.tight_layout()

save_path = f'{ARTIFACTS}/attack_examples/original_vs_pgd.png'
plt.savefig(save_path, dpi=110, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {save_path}')